# 🎥 1 The Foundation (Data Preparation & Indexing)

In [ ]:
# ============================================================
# 1. Install required packages
# ============================================================

!pip -q install pandas numpy openpyxl sentence-transformers faiss-cpu groq streamlit scikit-learn

In [ ]:
# ============================================================
# 2. Imports and configuration
# ============================================================

import os
import re
import json
import math
import pickle
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
import faiss

# Optional: only needed for LLM generation.
try:
    from groq import Groq
except Exception:
    Groq = None

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 100)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Use the uploaded filenames. The fallback paths make the notebook robust if files are renamed.
FLIGHT_FILE_CANDIDATES = [
    "KingAbdulazizAirport_Flight_Data(1).csv",
    "KingAbdulazizAirport_Flight_Data.csv",
]

TRAIN_FILE_CANDIDATES = [
    "HHR_Train (1)(1).xlsx",
    "HHR_Train (1).xlsx",
    "HHR_Train.xlsx",
]


def find_existing_file(candidates):
    for name in candidates:
        if Path(name).exists():
            return name
    # Colab/Jupyter sometimes uses /content; ChatGPT sandbox uses /mnt/data.
    for base in [Path("/mnt/data"), Path("/content")]:
        for name in candidates:
            candidate = base / name
            if candidate.exists():
                return str(candidate)
    raise FileNotFoundError(f"Could not find any of these files: {candidates}")

FLIGHT_PATH = find_existing_file(FLIGHT_FILE_CANDIDATES)
TRAIN_PATH = find_existing_file(TRAIN_FILE_CANDIDATES)

print("Flight dataset:", FLIGHT_PATH)
print("Train dataset:", TRAIN_PATH)

Flight dataset: KingAbdulazizAirport_Flight_Data.csv
Train dataset: HHR_Train (1).xlsx


In [ ]:
# ============================================================
# 3. Load datasets
# ============================================================

flights_raw = pd.read_csv(FLIGHT_PATH)
trains_raw = pd.read_excel(TRAIN_PATH)

print("Flights shape:", flights_raw.shape)
print("Trains shape:", trains_raw.shape)

display(flights_raw.head())
display(trains_raw.head())

Flights shape: (1810, 16)
Trains shape: (1350, 14)


,movement,flight_id,airline,origin,transit_point,destination,take_off_time,landing_time,transit_duration_mins,total_duration_mins,aircraft_model,carbon_kg,eco_price_sar,price_status,flight_date,day
0,DEPARTURE,EY 604,Etihad,JED,Direct,AUH,5:40,9:20,0,160,Airbus A321,166000.0,461.0,typical,8/22/2026,Saturday
1,DEPARTURE,SV 570,Saudia,JED,Direct,AUH,11:40,15:25,0,165,Airbus A320,176000.0,461.0,typical,8/22/2026,Saturday
2,DEPARTURE,SV 1020,Saudia,JED,RUH,AUH,7:00,14:05,150,365,Airbus A321,223000.0,471.0,typical,8/22/2026,Saturday
3,DEPARTURE,EY 602,Etihad,JED,Direct,AUH,3:10,6:55,0,165,Airbus A321,166000.0,552.0,typical,8/22/2026,Saturday
4,DEPARTURE,QR 1189,Qatar Airways,JED,DOH,AUH,22:35,4:25,80,290,Boeing 787,152000.0,697.0,typical,8/22/2026,Saturday


,Trip_ID,Departure_Station,Arrival_Station,Departure_Time,Arrival_Time,Trip_Date,Train_Model,Class_Type,Distance_KM,Occupancy_Rate_%,Seat_Status,Total_Fare_SAR,Efficiency_Score,Logistics_Note
0,HHR-MK-30000,Makkah,KAEC (Rabigh),12:45,13:39,2026-07-15,Talgo 350 SRO,Economy,186,96%,Confirmed,93.0,0.5,Long Haul
1,HHR-MK-30001,Madinah,Makkah,05:00,06:57,2026-06-02,Talgo 350 SRO,Economy,450,86%,Available,225.0,0.5,Long Haul
2,HHR-MK-30002,Makkah,KAEC (Rabigh),19:45,20:39,2026-06-12,Talgo 350 SRO,Economy,186,90%,Available,93.0,0.5,Long Haul
3,HHR-MK-30003,Makkah,Madinah,08:45,10:42,2026-04-18,Talgo 350 SRO,Economy,450,82%,Confirmed,225.0,0.5,Long Haul
4,HHR-MK-30004,Makkah,KAEC (Rabigh),05:15,06:09,2026-07-16,Talgo 350 SRO,Business,186,66%,Confirmed,204.6,1.1,Long Haul


In [ ]:
# ============================================================
# 4. Data cleaning and validation
# ============================================================

flights = flights_raw.copy()
trains = trains_raw.copy()

# Standardize whitespace in column names.
flights.columns = flights.columns.str.strip()
trains.columns = trains.columns.str.strip()

# Convert numeric columns safely.
flight_numeric_cols = ["transit_duration_mins", "total_duration_mins", "carbon_kg", "eco_price_sar"]
for col in flight_numeric_cols:
    flights[col] = pd.to_numeric(flights[col], errors="coerce")

train_numeric_cols = ["Distance_KM", "Total_Fare_SAR", "Efficiency_Score"]
for col in train_numeric_cols:
    trains[col] = pd.to_numeric(trains[col], errors="coerce")

# Convert occupancy from strings like "96%" to float 96.0.
trains["Occupancy_Rate_%"] = (
    trains["Occupancy_Rate_%"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
)
trains["Occupancy_Rate_%"] = pd.to_numeric(trains["Occupancy_Rate_%"], errors="coerce")

# Add row IDs for traceability.
flights = flights.reset_index(drop=True)
trains = trains.reset_index(drop=True)
flights["row_id"] = flights.index.astype(int)
trains["row_id"] = trains.index.astype(int)

print("Missing values in flights:")
display(flights.isna().sum().to_frame("missing"))

print("Missing values in trains:")
display(trains.isna().sum().to_frame("missing"))

Missing values in flights:


,missing
movement,0
flight_id,0
airline,0
origin,0
transit_point,0
destination,0
take_off_time,0
landing_time,0
transit_duration_mins,0
total_duration_mins,0


Missing values in trains:


,missing
Trip_ID,0
Departure_Station,0
Arrival_Station,0
Departure_Time,0
Arrival_Time,0
Trip_Date,0
Train_Model,0
Class_Type,0
Distance_KM,0
Occupancy_Rate_%,0


In [ ]:
# ============================================================
# 5. Basic domain analysis
# ============================================================

print("Unique flight origins:", sorted(flights["origin"].dropna().unique())[:40])
print("Unique flight destinations:", sorted(flights["destination"].dropna().unique())[:40])
print("Train stations:", sorted(set(trains["Departure_Station"]).union(set(trains["Arrival_Station"]))))

flight_summary = pd.DataFrame({
    "metric": ["number_of_flights", "number_of_airlines", "avg_flight_price_sar", "avg_flight_duration_mins", "avg_carbon_kg"],
    "value": [
        len(flights),
        flights["airline"].nunique(),
        round(flights["eco_price_sar"].mean(), 2),
        round(flights["total_duration_mins"].mean(), 2),
        round(flights["carbon_kg"].mean(), 2),
    ]
})

train_summary = pd.DataFrame({
    "metric": ["number_of_train_trips", "number_of_stations", "avg_train_fare_sar", "avg_occupancy_rate", "avg_distance_km"],
    "value": [
        len(trains),
        len(set(trains["Departure_Station"]).union(set(trains["Arrival_Station"]))),
        round(trains["Total_Fare_SAR"].mean(), 2),
        round(trains["Occupancy_Rate_%"].mean(), 2),
        round(trains["Distance_KM"].mean(), 2),
    ]
})

display(flight_summary)
display(train_summary)

Unique flight origins: ['AMM', 'AMS', 'AUH', 'BAH', 'BCN', 'BKK', 'BOM', 'CAI', 'CDG', 'DEL', 'DMM', 'DXB', 'FCO', 'GIZ', 'HKG', 'HND', 'IAD', 'ICN', 'IST', 'JED', 'JFK', 'KUL', 'KWI', 'LHR', 'MAD', 'MCT', 'MED', 'RUH', 'SAW', 'SIN', 'TIF', 'TUU', 'YNB', 'ZRH']
Unique flight destinations: ['AMM', 'AUH', 'BAH', 'BCN', 'BKK', 'BOM', 'CAI', 'CDG', 'DEL', 'DMM', 'DOH', 'DXB', 'FCO', 'FRA', 'GIZ', 'HKG', 'HND', 'HOF', 'IAD', 'ICN', 'IST', 'JED', 'JFK', 'KUL', 'KWI', 'LAX', 'LHR', 'MAD', 'MCT', 'MED', 'ORD', 'RUH', 'SAW', 'SIN', 'TUU', 'YNB', 'ZRH']
Train stations: ['Jeddah (Sulimaniyah)', 'KAEC (Rabigh)', 'KAIA (Airport)', 'Madinah', 'Makkah']


,metric,value
0,number_of_flights,1810.00
1,number_of_airlines,53.00
2,avg_flight_price_sar,1473.59
3,avg_flight_duration_mins,488.26
4,avg_carbon_kg,302028.18


,metric,value
0,number_of_train_trips,1350.00
1,number_of_stations,5.00
2,avg_train_fare_sar,143.35
3,avg_occupancy_rate,81.98
4,avg_distance_km,219.44


In [ ]:
# ============================================================
# 6. Convert rows into RAG documents
# ============================================================

CITY_TO_AIRPORT = {
    "Jeddah": "JED",
    "Riyadh": "RUH",
    "Dammam": "DMM",
    "Madinah": "MED",
    "Medina": "MED",
    "Dubai": "DXB",
    "Abu Dhabi": "AUH",
    "Doha": "DOH",
    "London": "LHR",
    "New York": "JFK",
    "Kuala Lumpur": "KUL",
    "Amman": "AMM",
}
AIRPORT_TO_CITY = {v: k for k, v in CITY_TO_AIRPORT.items()}

CITY_ALIASES = {
    "jeddah": "Jeddah",
    "riyadh": "Riyadh",
    "dammam": "Dammam",
    "makkah": "Makkah",
    "mecca": "Makkah",
    "madinah": "Madinah",
    "medina": "Madinah",
    "kaia": "KAIA (Airport)",
    "airport": "KAIA (Airport)",
}


def safe_float(value, default=np.nan):
    """Convert dataset values into floats while preserving missing values."""
    try:
        return float(value)
    except Exception:
        return default


def minutes_from_time_for_docs(value):
    """Convert HH:MM time strings into minutes after midnight for route-document construction."""
    try:
        parsed = datetime.strptime(str(value), "%H:%M")
        return parsed.hour * 60 + parsed.minute
    except Exception:
        try:
            parsed = datetime.strptime(str(value), "%I:%M %p")
            return parsed.hour * 60 + parsed.minute
        except Exception:
            return np.nan


def duration_between_times_for_docs(start_time, end_time):
    """Compute a positive duration in minutes, allowing for next-day arrival."""
    start_mins = minutes_from_time_for_docs(start_time)
    end_mins = minutes_from_time_for_docs(end_time)
    if np.isnan(start_mins) or np.isnan(end_mins):
        return np.nan
    duration = end_mins - start_mins
    if duration < 0:
        duration += 24 * 60
    return duration


def flight_row_to_document(row):
    """Represent one flight row as retrieval evidence with explicit city, code, cost, time, and carbon terms."""
    origin_city = AIRPORT_TO_CITY.get(row["origin"], row["origin"])
    destination_city = AIRPORT_TO_CITY.get(row["destination"], row["destination"])
    text = (
        f"Flight evidence {row['row_id']}: flight {row['flight_id']} operated by {row['airline']} "
        f"travels from {origin_city} airport code {row['origin']} to {destination_city} airport code {row['destination']}. "
        f"The flight departs at {row['take_off_time']} and lands at {row['landing_time']}. "
        f"The total duration is {row['total_duration_mins']} minutes. "
        f"The economy price is {row['eco_price_sar']} SAR. "
        f"The estimated flight carbon emission is {row['carbon_kg']} kg. "
        f"The travel day is {row['day']} and the flight date is {row['flight_date']}. "
        f"This document is relevant for flight search, airport travel, price comparison, time comparison, and carbon comparison."
    )
    metadata = {
        "doc_type": "flight",
        "row_id": int(row["row_id"]),
        "flight_id": row["flight_id"],
        "airline": row["airline"],
        "origin": row["origin"],
        "destination": row["destination"],
        "origin_city": origin_city,
        "destination_city": destination_city,
        "take_off_time": row["take_off_time"],
        "landing_time": row["landing_time"],
        "duration_mins": safe_float(row["total_duration_mins"]),
        "price_sar": safe_float(row["eco_price_sar"]),
        "carbon_kg": safe_float(row["carbon_kg"]),
        "day": row["day"],
    }
    return {"doc_id": f"flight_{int(row['row_id'])}", "source": "flight", "text": text, "metadata": metadata}


def train_row_to_document(row):
    """Represent one train row as retrieval evidence with explicit station, fare, occupancy, and duration terms."""
    train_duration_mins = duration_between_times_for_docs(row["Departure_Time"], row["Arrival_Time"])
    text = (
        f"Train evidence {row['row_id']}: HHR train trip {row['Trip_ID']} travels from {row['Departure_Station']} "
        f"to {row['Arrival_Station']}. The train departs at {row['Departure_Time']} and arrives at {row['Arrival_Time']}. "
        f"The estimated train duration is {train_duration_mins} minutes. "
        f"The fare is {row['Total_Fare_SAR']} SAR. "
        f"The occupancy rate is {row['Occupancy_Rate_%']} percent. "
        f"The distance is {row['Distance_KM']} km. "
        f"This document is relevant for HHR train search, station travel, fare comparison, crowding, occupancy, and route planning."
    )
    metadata = {
        "doc_type": "train",
        "row_id": int(row["row_id"]),
        "trip_id": row["Trip_ID"],
        "departure_station": row["Departure_Station"],
        "arrival_station": row["Arrival_Station"],
        "departure_time": row["Departure_Time"],
        "arrival_time": row["Arrival_Time"],
        "duration_mins": safe_float(train_duration_mins),
        "fare_sar": safe_float(row["Total_Fare_SAR"]),
        "occupancy_rate": safe_float(row["Occupancy_Rate_%"]),
        "distance_km": safe_float(row["Distance_KM"]),
    }
    return {"doc_id": f"train_{int(row['row_id'])}", "source": "train", "text": text, "metadata": metadata}


def build_connected_route_table(origin_city, final_destination, min_connection_minutes=45):
    """Create feasible flight-to-Jeddah plus HHR route candidates for retrieval evidence."""
    origin_code = CITY_TO_AIRPORT.get(origin_city)
    if origin_code is None or final_destination not in ["Makkah", "Madinah"]:
        return pd.DataFrame()

    flight_options = flights[(flights["origin"] == origin_code) & (flights["destination"] == "JED")].copy()
    train_options = trains[(trains["Departure_Station"] == "KAIA (Airport)") & (trains["Arrival_Station"] == final_destination)].copy()

    rows = []
    for _, flight in flight_options.iterrows():
        landing_mins = minutes_from_time_for_docs(flight["landing_time"])
        for _, train in train_options.iterrows():
            train_departure_mins = minutes_from_time_for_docs(train["Departure_Time"])
            if np.isnan(landing_mins) or np.isnan(train_departure_mins):
                continue

            wait_mins = train_departure_mins - landing_mins
            if wait_mins < 0:
                wait_mins += 24 * 60
            if wait_mins < min_connection_minutes:
                continue

            train_duration_mins = duration_between_times_for_docs(train["Departure_Time"], train["Arrival_Time"])
            if np.isnan(train_duration_mins):
                continue

            rows.append({
                "origin_city": origin_city,
                "final_destination": final_destination,
                "flight_id": flight["flight_id"],
                "airline": flight["airline"],
                "flight_origin": flight["origin"],
                "flight_destination": flight["destination"],
                "take_off_time": flight["take_off_time"],
                "landing_time": flight["landing_time"],
                "flight_duration_mins": safe_float(flight["total_duration_mins"]),
                "flight_price_sar": safe_float(flight["eco_price_sar"]),
                "flight_carbon_kg": safe_float(flight["carbon_kg"]),
                "train_trip_id": train["Trip_ID"],
                "train_from": train["Departure_Station"],
                "train_to": train["Arrival_Station"],
                "train_departure_time": train["Departure_Time"],
                "train_arrival_time": train["Arrival_Time"],
                "train_duration_mins": safe_float(train_duration_mins),
                "train_fare_sar": safe_float(train["Total_Fare_SAR"]),
                "train_occupancy_rate": safe_float(train["Occupancy_Rate_%"]),
                "wait_mins": safe_float(wait_mins),
                "total_duration_mins": safe_float(flight["total_duration_mins"]) + safe_float(wait_mins) + safe_float(train_duration_mins),
                "total_cost_sar": safe_float(flight["eco_price_sar"]) + safe_float(train["Total_Fare_SAR"]),
                "total_carbon_kg": safe_float(flight["carbon_kg"]),
            })

    return pd.DataFrame(rows)


def connected_route_to_document(row, preference_label, route_rank):
    """Represent one complete multi-leg route as a single retrieval unit."""
    text = (
        f"Connected route evidence {route_rank}: {preference_label} route from {row['origin_city']} to {row['final_destination']} "
        f"using flight plus HHR train. Flight {row['flight_id']} by {row['airline']} travels from {row['flight_origin']} to JED, "
        f"departing at {row['take_off_time']} and landing at {row['landing_time']}. "
        f"Then HHR train {row['train_trip_id']} travels from {row['train_from']} to {row['train_to']}, "
        f"departing at {row['train_departure_time']} and arriving at {row['train_arrival_time']}. "
        f"The connection waiting time is {row['wait_mins']:.0f} minutes. "
        f"The total route duration is {row['total_duration_mins']:.0f} minutes including flight, wait, and train. "
        f"The total route cost is {row['total_cost_sar']:.2f} SAR. "
        f"The flight carbon emission is {row['total_carbon_kg']:.2f} kg. "
        f"The train occupancy rate is {row['train_occupancy_rate']:.1f} percent. "
        f"This document is relevant for cheapest route, fastest route, lowest carbon route, least crowded route, and connected route planning."
    )
    metadata = {
        "doc_type": "connected_route",
        "origin_city": row["origin_city"],
        "destination": row["final_destination"],
        "preference_label": preference_label,
        "route_rank": int(route_rank),
        "flight_id": row["flight_id"],
        "train_trip_id": row["train_trip_id"],
        "flight_origin": row["flight_origin"],
        "flight_destination": row["flight_destination"],
        "train_from": row["train_from"],
        "train_to": row["train_to"],
        "total_cost_sar": safe_float(row["total_cost_sar"]),
        "total_duration_mins": safe_float(row["total_duration_mins"]),
        "total_carbon_kg": safe_float(row["total_carbon_kg"]),
        "wait_mins": safe_float(row["wait_mins"]),
        "train_occupancy_rate": safe_float(row["train_occupancy_rate"]),
    }
    doc_id = f"route_{row['origin_city'].lower()}_{row['final_destination'].lower()}_{preference_label}_{route_rank}"
    return {"doc_id": doc_id, "source": "route", "text": text, "metadata": metadata}


def build_connected_route_documents(top_n_per_preference=15):
    """Add route-level evidence so that multi-leg questions retrieve complete answers instead of isolated rows."""
    route_documents = []
    supported_origin_cities = [
        city for city, code in CITY_TO_AIRPORT.items()
        if code in set(flights.loc[flights["destination"] == "JED", "origin"])
    ]

    preference_sorting = {
        "cheapest": ["total_cost_sar", "total_duration_mins"],
        "fastest": ["total_duration_mins", "total_cost_sar"],
        "lowest_carbon": ["total_carbon_kg", "total_cost_sar"],
        "least_crowded": ["train_occupancy_rate", "total_cost_sar"],
    }

    for origin_city in supported_origin_cities:
        for final_destination in ["Makkah", "Madinah"]:
            routes_df = build_connected_route_table(origin_city, final_destination)
            if routes_df.empty:
                continue

            for preference_label, sort_cols in preference_sorting.items():
                ranked_routes = routes_df.sort_values(sort_cols).head(top_n_per_preference).reset_index(drop=True)
                for route_rank, (_, route_row) in enumerate(ranked_routes.iterrows(), start=1):
                    route_documents.append(connected_route_to_document(route_row, preference_label, route_rank))

    return route_documents


flight_documents = [flight_row_to_document(row) for _, row in flights.iterrows()]
train_documents = [train_row_to_document(row) for _, row in trains.iterrows()]
route_documents = build_connected_route_documents(top_n_per_preference=15)

documents = flight_documents + train_documents + route_documents
texts = [doc["text"] for doc in documents]

print("Flight documents:", len(flight_documents))
print("Train documents:", len(train_documents))
print("Connected route documents:", len(route_documents))
print("Total RAG documents:", len(documents))
print(texts[0][:700])

Flight documents: 1810
Train documents: 1350
Connected route documents: 600
Total RAG documents: 3760
Flight evidence 0: flight EY 604 operated by Etihad travels from Jeddah airport code JED to Abu Dhabi airport code AUH. The flight departs at 5:40 and lands at 9:20. The total duration is 160 minutes. The economy price is 461.0 SAR. The estimated flight carbon emission is 166000.0 kg. The travel day is Saturday and the flight date is 8/22/2026. This document is relevant for flight search, airport travel, price comparison, time comparison, and carbon comparison.


In [ ]:
# ============================================================
# 7. Build embeddings and FAISS index
# ============================================================
# The retriever uses normalized sentence embeddings and FAISS inner-product search.
# Normalized vectors make the inner product equivalent to cosine similarity.

EMBEDDING_MODEL_CANDIDATES = [
    "BAAI/bge-small-en-v1.5",
    "sentence-transformers/all-MiniLM-L6-v2",
]

embedding_model = None
last_embedding_error = None

for model_name in EMBEDDING_MODEL_CANDIDATES:
    try:
        EMBEDDING_MODEL_NAME = model_name
        embedding_model = SentenceTransformer(model_name)
        break
    except Exception as error:
        last_embedding_error = error

if embedding_model is None:
    raise RuntimeError(f"No embedding model could be loaded. Last error: {last_embedding_error}")

def format_query_for_embedding(query):
    """Use the recommended BGE query prefix when the selected model belongs to the BGE family."""
    if "bge" in EMBEDDING_MODEL_NAME.lower():
        return "Represent this sentence for searching relevant passages: " + str(query)
    return str(query)

def format_document_for_embedding(text):
    """Keep document formatting model-aware while preserving compatibility with non-BGE models."""
    return str(text)

embeddings = embedding_model.encode(
    [format_document_for_embedding(text) for text in texts],
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding dimension:", embedding_dim)
print("Documents indexed:", index.ntotal)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/59 [00:00<?, ?it/s]

Embedding model: BAAI/bge-small-en-v1.5
Embedding dimension: 384
Documents indexed: 3760


# 🎥  2: The Engine (Retrieval, Routing & LLM Generation)


In [ ]:
# ============================================================
# 8. Hybrid semantic retriever
# ============================================================
# The retrieval layer combines dense semantic similarity with lightweight lexical and metadata scoring.
# This improves route queries because route planning needs exact city, station, cost, duration, and carbon constraints.

def retrieval_intent(query):
    """Classify the query into the evidence type most likely to answer it."""
    q = normalize_query_text(query) if "normalize_query_text" in globals() else re.sub(r"\s+", " ", str(query).lower()).strip()

    if any(term in q for term in ["cheapest", "cheap", "fastest", "fast", "lowest carbon", "carbon", "eco", "route", "way from"]):
        if any(city in q for city in ["makkah", "mecca", "madinah", "medina"]):
            return "connected_route"
    if any(term in q for term in ["train", "hhr", "station", "occupancy", "crowded"]):
        return "train"
    if any(term in q for term in ["flight", "airport", "airline", "peak days"]):
        return "flight"
    return "general"


def rewrite_query_for_retrieval(query):
    """Expand short user questions into retrieval-oriented wording while preserving the original meaning."""
    q = normalize_query_text(query) if "normalize_query_text" in globals() else re.sub(r"\s+", " ", str(query).lower()).strip()
    origin, destination = extract_cities(query) if "extract_cities" in globals() else (None, None)
    preference = detect_preference(query) if "detect_preference" in globals() else "balanced"

    if origin and destination in ["Makkah", "Madinah"]:
        if preference == "cheapest":
            return f"cheapest connected route from {origin} to {destination} using flight to Jeddah JED and HHR train from KAIA Airport with total cost SAR"
        if preference == "fastest":
            return f"fastest connected route from {origin} to {destination} using flight to Jeddah JED and HHR train from KAIA Airport with total duration waiting time minutes"
        if preference == "lowest_carbon":
            return f"lowest carbon eco friendly connected route from {origin} to {destination} using flight to Jeddah JED and HHR train from KAIA Airport carbon emission kg"
        if preference == "least_crowded":
            return f"least crowded connected route from {origin} to {destination} using flight to Jeddah JED and HHR train occupancy rate"
        return f"best connected route from {origin} to {destination} using flight to Jeddah JED and HHR train from KAIA Airport"

    if "kaia" in q and "makkah" in q:
        return "available HHR trains from KAIA Airport to Makkah departure arrival fare occupancy"
    if "riyadh" in q and "jeddah" in q and "flight" in q:
        return "available flights from Riyadh RUH to Jeddah JED departure landing price carbon"

    return str(query)


def token_set(text):
    """Tokenize text for deterministic lexical matching."""
    return set(re.findall(r"[a-zA-Z0-9]+", str(text).lower()))


def keyword_overlap_score(query, document_text):
    """Compute a normalized keyword-overlap score between the query and one candidate document."""
    query_tokens = token_set(query)
    document_tokens = token_set(document_text)
    if not query_tokens:
        return 0.0
    return len(query_tokens.intersection(document_tokens)) / len(query_tokens)


def metadata_constraint_score(query, document):
    """Reward candidates whose metadata matches the city and intent constraints stated in the query."""
    q = normalize_query_text(query) if "normalize_query_text" in globals() else re.sub(r"\s+", " ", str(query).lower()).strip()
    meta = document["metadata"]
    source = document["source"]
    score = 0.0

    if source == "route":
        if "riyadh" in q and meta.get("origin_city") == "Riyadh":
            score += 0.30
        if "dammam" in q and meta.get("origin_city") == "Dammam":
            score += 0.30
        if ("madinah" in q or "medina" in q) and meta.get("origin_city") == "Madinah":
            score += 0.30
        if ("makkah" in q or "mecca" in q) and meta.get("destination") == "Makkah":
            score += 0.30
        if ("madinah" in q or "medina" in q) and meta.get("destination") == "Madinah":
            score += 0.30
        if "cheap" in q and meta.get("preference_label") == "cheapest":
            score += 0.35
        if "fast" in q and meta.get("preference_label") == "fastest":
            score += 0.35
        if "carbon" in q and meta.get("preference_label") == "lowest_carbon":
            score += 0.35
        if "crowd" in q and meta.get("preference_label") == "least_crowded":
            score += 0.35

    if source == "flight":
        if "riyadh" in q and meta.get("origin") == "RUH":
            score += 0.20
        if "jeddah" in q and meta.get("destination") == "JED":
            score += 0.20

    if source == "train":
        if "kaia" in q and meta.get("departure_station") == "KAIA (Airport)":
            score += 0.20
        if ("makkah" in q or "mecca" in q) and meta.get("arrival_station") == "Makkah":
            score += 0.20
        if ("madinah" in q or "medina" in q) and meta.get("arrival_station") == "Madinah":
            score += 0.20

    return min(score, 1.0)


def infer_source_filter(query):
    """Limit the candidate pool to the most appropriate evidence type before final ranking."""
    intent = retrieval_intent(query)
    if intent == "connected_route":
        return ["route"]
    if intent == "flight":
        return ["flight"]
    if intent == "train":
        return ["train"]
    return None


def semantic_retrieve(query, top_k=10, source_filter=None, candidate_k=120):
    """Retrieve and rerank evidence using semantic similarity, keyword overlap, and metadata constraints."""
    rewritten_query = rewrite_query_for_retrieval(query)
    query_embedding = embedding_model.encode(
        [format_query_for_embedding(rewritten_query)],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    active_source_filter = source_filter if source_filter is not None else infer_source_filter(query)
    allowed_sources = set(active_source_filter) if active_source_filter else None

    search_k = min(max(candidate_k, top_k * 10), len(documents))
    scores, indices = index.search(query_embedding, search_k)

    candidates = []
    for semantic_score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        doc = documents[int(idx)]
        if allowed_sources and doc["source"] not in allowed_sources:
            continue

        lexical_score = keyword_overlap_score(rewritten_query, doc["text"])
        metadata_score = metadata_constraint_score(query, doc)
        final_score = (0.45 * float(semantic_score)) + (0.20 * lexical_score) + (0.35 * metadata_score)

        candidates.append({
            "score": float(final_score),
            "semantic_score": float(semantic_score),
            "keyword_score": float(lexical_score),
            "metadata_score": float(metadata_score),
            "doc_id": doc["doc_id"],
            "source": doc["source"],
            "text": doc["text"],
            "metadata": doc["metadata"],
        })

    ranked = sorted(candidates, key=lambda item: item["score"], reverse=True)[:top_k]

    for rank, item in enumerate(ranked, start=1):
        item["rank"] = rank

    return ranked


# Smoke test: the top results for a multi-leg question should now be complete connected-route documents.
sample_results = semantic_retrieve("What is the cheapest way from Riyadh to Makkah?", top_k=5)
for result in sample_results:
    print(result["rank"], round(result["score"], 4), result["doc_id"], result["source"])
    print(result["text"][:350], "...\n")

1 0.8567 route_riyadh_makkah_cheapest_1 route
Connected route evidence 1: cheapest route from Riyadh to Makkah using flight plus HHR train. Flight XY 19 by Flynas travels from RUH to JED, departing at 13:10 and landing at 15:00. Then HHR train HHR-MK-30051 travels from KAIA (Airport) to Makkah, departing at 15:45 and arriving at 16:17. The connection waiting time is 45 minutes. The total route ...

2 0.8539 route_riyadh_makkah_cheapest_6 route
Connected route evidence 6: cheapest route from Riyadh to Makkah using flight plus HHR train. Flight XY 19 by Flynas travels from RUH to JED, departing at 13:10 and landing at 15:00. Then HHR train HHR-MK-30543 travels from KAIA (Airport) to Makkah, departing at 16:00 and arriving at 16:32. The connection waiting time is 60 minutes. The total route ...

3 0.8536 route_riyadh_makkah_cheapest_7 route
Connected route evidence 7: cheapest route from Riyadh to Makkah using flight plus HHR train. Flight XY 19 by Flynas travels from RUH to JED, departin

In [ ]:
# ============================================================
# 9. Structured route helpers
# ============================================================


def normalize_query_text(text):
    return re.sub(r"\s+", " ", str(text).lower()).strip()


def extract_cities(query):
    """Extract origin and final destination from a query using dataset-supported aliases."""
    q = normalize_query_text(query)
    found = []
    for alias, city in CITY_ALIASES.items():
        if re.search(rf"\b{re.escape(alias)}\b", q) and city not in found:
            found.append(city)

    # Heuristic: for travel questions, first city is origin, last city is destination.
    if len(found) >= 2:
        return found[0], found[-1]
    if len(found) == 1:
        return None, found[0]
    return None, None


def detect_preference(query):
    q = normalize_query_text(query)
    if any(w in q for w in ["cheap", "cheapest", "budget", "low cost", "affordable", "price"]):
        return "cheapest"
    if any(w in q for w in ["fast", "fastest", "quick", "shortest", "time"]):
        return "fastest"
    if any(w in q for w in ["carbon", "sustainable", "eco", "environment", "emission", "green"]):
        return "lowest_carbon"
    if any(w in q for w in ["crowded", "busy", "occupancy"]):
        return "least_crowded"
    return "balanced"


def time_to_minutes(value):
    """Convert HH:MM time to minutes after midnight."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    for fmt in ["%H:%M", "%I:%M %p"]:
        try:
            dt = datetime.strptime(value, fmt)
            return dt.hour * 60 + dt.minute
        except ValueError:
            pass
    return np.nan


def duration_between_minutes(start_time, end_time):
    start = time_to_minutes(start_time)
    end = time_to_minutes(end_time)
    if np.isnan(start) or np.isnan(end):
        return np.nan
    if end < start:
        end += 24 * 60
    return end - start


def find_flights(origin_city=None, destination_code="JED"):
    if origin_city is None:
        return pd.DataFrame()
    origin_code = CITY_TO_AIRPORT.get(origin_city, origin_city)
    return flights[(flights["origin"] == origin_code) & (flights["destination"] == destination_code)].copy()


def find_trains(from_station, to_station):
    return trains[(trains["Departure_Station"] == from_station) & (trains["Arrival_Station"] == to_station)].copy()

In [ ]:
# ============================================================
# 10. Connected flight + HHR route planner
# ============================================================


def recommend_connected_route(origin_city, final_destination, preference="balanced", min_connection_minutes=45):
    """Recommend a valid flight-to-Jeddah + HHR train route."""
    if final_destination not in ["Makkah", "Madinah"]:
        return None, "This prototype supports final HHR destinations of Makkah and Madinah."

    flight_options = find_flights(origin_city, "JED")
    train_options = find_trains("KAIA (Airport)", final_destination)

    if flight_options.empty:
        return None, f"No flight records found from {origin_city} to Jeddah (JED)."
    if train_options.empty:
        return None, f"No HHR train records found from KAIA (Airport) to {final_destination}."

    connected_routes = []
    for _, flight in flight_options.iterrows():
        landing_mins = time_to_minutes(flight["landing_time"])
        for _, train in train_options.iterrows():
            train_depart_mins = time_to_minutes(train["Departure_Time"])
            if np.isnan(landing_mins) or np.isnan(train_depart_mins):
                continue

            # Handle next-day connections if needed.
            wait_mins = train_depart_mins - landing_mins
            if wait_mins < 0:
                wait_mins += 24 * 60

            if wait_mins < min_connection_minutes:
                continue

            train_duration_mins = duration_between_minutes(train["Departure_Time"], train["Arrival_Time"])
            total_duration_mins = float(flight["total_duration_mins"]) + wait_mins + train_duration_mins
            total_cost_sar = float(flight["eco_price_sar"]) + float(train["Total_Fare_SAR"])
            carbon_kg = float(flight["carbon_kg"])

            connected_routes.append({
                "flight_id": flight["flight_id"],
                "airline": flight["airline"],
                "origin_city": origin_city,
                "flight_origin": flight["origin"],
                "flight_destination": flight["destination"],
                "take_off_time": flight["take_off_time"],
                "landing_time": flight["landing_time"],
                "flight_duration_mins": float(flight["total_duration_mins"]),
                "flight_price_sar": float(flight["eco_price_sar"]),
                "flight_carbon_kg": carbon_kg,
                "train_trip_id": train["Trip_ID"],
                "train_from": train["Departure_Station"],
                "train_to": train["Arrival_Station"],
                "train_departure_time": train["Departure_Time"],
                "train_arrival_time": train["Arrival_Time"],
                "train_duration_mins": float(train_duration_mins),
                "train_fare_sar": float(train["Total_Fare_SAR"]),
                "train_occupancy_rate": float(train["Occupancy_Rate_%"]),
                "wait_mins": float(wait_mins),
                "total_duration_mins": float(total_duration_mins),
                "total_cost_sar": float(total_cost_sar),
                "total_carbon_kg": carbon_kg,
            })

    if not connected_routes:
        return None, "No feasible connected route found with the minimum connection time."

    routes_df = pd.DataFrame(connected_routes)

    if preference == "cheapest":
        best = routes_df.sort_values(["total_cost_sar", "total_duration_mins"]).iloc[0]
    elif preference == "fastest":
        best = routes_df.sort_values(["total_duration_mins", "total_cost_sar"]).iloc[0]
    elif preference == "lowest_carbon":
        best = routes_df.sort_values(["total_carbon_kg", "total_cost_sar"]).iloc[0]
    elif preference == "least_crowded":
        best = routes_df.sort_values(["train_occupancy_rate", "total_cost_sar"]).iloc[0]
    else:
        # Balanced score with normalized cost, duration, and occupancy.
        temp = routes_df.copy()
        for col in ["total_cost_sar", "total_duration_mins", "train_occupancy_rate"]:
            denom = temp[col].max() - temp[col].min()
            temp[col + "_norm"] = 0 if denom == 0 else (temp[col] - temp[col].min()) / denom
        temp["balanced_score"] = 0.4 * temp["total_cost_sar_norm"] + 0.4 * temp["total_duration_mins_norm"] + 0.2 * temp["train_occupancy_rate_norm"]
        best = temp.sort_values("balanced_score").iloc[0]

    explanation = (
        f"Recommended connected route from {origin_city} to {final_destination}: "
        f"take flight {best['flight_id']} with {best['airline']} from {best['flight_origin']} to JED, "
        f"departing at {best['take_off_time']} and landing at {best['landing_time']}. "
        f"Then take HHR train {best['train_trip_id']} from {best['train_from']} to {best['train_to']}, "
        f"departing at {best['train_departure_time']} and arriving at {best['train_arrival_time']}. "
        f"Connection wait is {best['wait_mins']:.0f} minutes. "
        f"Total estimated duration is {best['total_duration_mins']:.0f} minutes including flight, wait, and train. "
        f"Total estimated cost is {best['total_cost_sar']:.2f} SAR. "
        f"Flight carbon emission is {best['total_carbon_kg']:.2f} kg. "
        f"Train occupancy rate is {best['train_occupancy_rate']:.1f}%. "
        f"Selection preference: {preference}."
    )

    return best.to_dict(), explanation

# Smoke test
route, route_explanation = recommend_connected_route("Riyadh", "Makkah", "cheapest")
print(route_explanation)

Recommended connected route from Riyadh to Makkah: take flight XY 19 with Flynas from RUH to JED, departing at 13:10 and landing at 15:00. Then take HHR train HHR-MK-30051 from KAIA (Airport) to Makkah, departing at 15:45 and arriving at 16:17. Connection wait is 45 minutes. Total estimated duration is 187 minutes including flight, wait, and train. Total estimated cost is 386.50 SAR. Flight carbon emission is 76000.00 kg. Train occupancy rate is 93.0%. Selection preference: cheapest.


In [ ]:
# ============================================================
# 11. Hybrid retrieval context builder
# ============================================================
# The context builder keeps the original structured evidence layer and adds the improved retriever.
# This preserves the system behavior while improving the quality of retrieved evidence.

def structured_retrieve(query):
    origin, destination = extract_cities(query)
    preference = detect_preference(query)

    q = normalize_query_text(query)

    # Aggregation questions are answered from structured data because they require counting or grouping.
    if any(w in q for w in ["peak days", "busy days", "flights by day"]):
        peak_days = flights["day"].value_counts().reset_index()
        peak_days.columns = ["day", "number_of_flights"]
        return {
            "type": "flight_peak_days",
            "preference": preference,
            "table": peak_days.head(10),
            "text": "Peak flight days by number of flights:\n" + peak_days.head(10).to_string(index=False)
        }

    if any(w in q for w in ["crowded", "occupancy", "busy hhr", "busy train"]):
        crowded = trains.groupby(["Departure_Station", "Arrival_Station"], as_index=False)["Occupancy_Rate_%"].mean()
        crowded = crowded.sort_values("Occupancy_Rate_%", ascending=False)
        return {
            "type": "crowded_train_routes",
            "preference": preference,
            "table": crowded.head(10),
            "text": "Most crowded HHR routes by average occupancy:\n" + crowded.head(10).to_string(index=False)
        }

    # Route questions still use structured computation for exact cost, duration, wait time, and carbon values.
    if origin and destination in ["Makkah", "Madinah"]:
        best_route, explanation = recommend_connected_route(origin, destination, preference)
        return {
            "type": "connected_route",
            "origin": origin,
            "destination": destination,
            "preference": preference,
            "route": best_route,
            "text": explanation,
        }

    return None


def build_rag_context(query, top_k=5):
    semantic_results = semantic_retrieve(query, top_k=top_k)
    structured_result = structured_retrieve(query)

    context_parts = []

    if structured_result is not None:
        context_parts.append("[STRUCTURED EVIDENCE]\n" + structured_result["text"])

    context_parts.append("[RETRIEVED RAG EVIDENCE]")
    for result in semantic_results:
        context_parts.append(
            f"Evidence ID: {result['doc_id']} | Source: {result['source']} | "
            f"Final score: {result['score']:.4f} | Semantic: {result.get('semantic_score', 0):.4f} | "
            f"Keyword: {result.get('keyword_score', 0):.4f} | Metadata: {result.get('metadata_score', 0):.4f}\n"
            f"{result['text']}"
        )

    return {
        "query": query,
        "context": "\n\n".join(context_parts),
        "semantic_results": semantic_results,
        "structured_result": structured_result,
    }


context_pack = build_rag_context("What is the cheapest way from Riyadh to Makkah?", top_k=5)
print(context_pack["context"][:2500])

[STRUCTURED EVIDENCE]
Recommended connected route from Riyadh to Makkah: take flight XY 19 with Flynas from RUH to JED, departing at 13:10 and landing at 15:00. Then take HHR train HHR-MK-30051 from KAIA (Airport) to Makkah, departing at 15:45 and arriving at 16:17. Connection wait is 45 minutes. Total estimated duration is 187 minutes including flight, wait, and train. Total estimated cost is 386.50 SAR. Flight carbon emission is 76000.00 kg. Train occupancy rate is 93.0%. Selection preference: cheapest.

[RETRIEVED RAG EVIDENCE]

Evidence ID: route_riyadh_makkah_cheapest_1 | Source: route | Final score: 0.9248 | Semantic: 0.9163 | Keyword: 0.9000 | Metadata: 0.9500
Connected route evidence 1: cheapest route from Riyadh to Makkah using flight plus HHR train. Flight XY 19 by Flynas travels from RUH to JED, departing at 13:10 and landing at 15:00. Then HHR train HHR-MK-30051 travels from KAIA (Airport) to Makkah, departing at 15:45 and arriving at 16:17. The connection waiting time is 4

In [ ]:
# ============================================================
# 12. Grounded answer generator
# ============================================================

import os

GENERATION_MODEL_NAME = "llama-3.1-8b-instant"

os.environ["GROQ_API_KEY"] = "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"

def get_groq_client():
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key or Groq is None:
        return None
    return Groq(api_key=api_key)


def extractive_fallback_answer(context_pack):
    """Safe fallback when no LLM API key is available."""
    query = context_pack["query"]
    structured = context_pack["structured_result"]
    semantic = context_pack["semantic_results"]

    if structured is not None:
        return "Based on the available dataset, " + structured["text"]

    evidence_lines = []
    for r in semantic[:3]:
        evidence_lines.append(f"- {r['doc_id']} ({r['source']}): {r['text']}")
    return (
        "Based on the available dataset, I found these relevant records:\n" +
        "\n".join(evidence_lines)
    )


def generate_answer(query, top_k=10, temperature=0.1):
    context_pack = build_rag_context(query, top_k=top_k)
    client = get_groq_client()

    if client is None:
        return extractive_fallback_answer(context_pack), context_pack

    system_prompt = """
You are a Smart Multi-Modal Travel Assistant for Saudi logistics and travel planning.

Rules:
1. Answer only using the provided retrieved evidence.
2. Do not invent prices, times, routes, emissions, or availability.
3. Start with: "Based on the available dataset,"
4. For route questions, include flight details, train details, total cost, total duration, wait time, and carbon emission when present.
5. If evidence is insufficient, say exactly what is missing.
6. Keep the answer concise and practical.
""".strip()

    user_prompt = f"""
User question:
{query}

Retrieved evidence:
{context_pack['context']}

Generate a grounded answer.
""".strip()

    response = client.chat.completions.create(
        model=GENERATION_MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=700,
    )

    return response.choices[0].message.content, context_pack
answer, pack = generate_answer("What is the cheapest way from Riyadh to Makkah?", top_k=10)
print(answer)

Based on the available dataset, the cheapest way from Riyadh to Makkah is using flight XY 19 by Flynas from RUH to JED, departing at 13:10 and landing at 15:00, followed by HHR train HHR-MK-30051 from KAIA (Airport) to Makkah, departing at 15:45 and arriving at 16:17. The connection waiting time is 45 minutes. The total route duration is 187 minutes including flight, wait, and train. The total route cost is 386.50 SAR. The flight carbon emission is 76000.00 kg. The train occupancy rate is 93.0 percent.


# 🎥 Speaker 3: Validation & Demonstration (Evaluation & UI)
*Focus: Proving the system works accurately and showing the final interactive product.*

In [ ]:
# ============================================================
# 13. Evaluation query set
# ============================================================

evaluation_queries = [
    {
        "question": "What are the available flights from Riyadh to Jeddah?",
        "intent": "flight_search",
        "expected": {"source": "flight", "origin": "RUH", "destination": "JED"},
        "required_terms": ["flight", "Riyadh", "Jeddah"],
    },
    {
        "question": "What are the available trains from KAIA Airport to Makkah?",
        "intent": "train_search",
        "expected": {"source": "train", "departure_station": "KAIA (Airport)", "arrival_station": "Makkah"},
        "required_terms": ["train", "KAIA", "Makkah"],
    },
    {
        "question": "What is the cheapest way from Riyadh to Makkah?",
        "intent": "connected_route",
        "expected": {"origin_city": "Riyadh", "destination": "Makkah", "preference": "cheapest"},
        "required_terms": ["flight", "train", "SAR", "cost", "Riyadh", "Makkah"],
    },
    {
        "question": "What is the fastest route from Riyadh to Makkah?",
        "intent": "connected_route",
        "expected": {"origin_city": "Riyadh", "destination": "Makkah", "preference": "fastest"},
        "required_terms": ["flight", "train", "duration", "minutes"],
    },
    {
        "question": "What is the lowest carbon option from Riyadh to Makkah?",
        "intent": "connected_route",
        "expected": {"origin_city": "Riyadh", "destination": "Makkah", "preference": "lowest_carbon"},
        "required_terms": ["carbon", "kg", "flight", "train"],
    },
    {
        "question": "Which HHR routes are most crowded?",
        "intent": "crowding",
        "expected": {"source": "train", "metric": "Occupancy_Rate_%"},
        "required_terms": ["occupancy", "crowded", "%"],
    },
    {
        "question": "What are the peak days for flights?",
        "intent": "flight_peak_days",
        "expected": {"source": "flight", "metric": "day"},
        "required_terms": ["flight", "day"],
    },
]

pd.DataFrame(evaluation_queries)

,question,intent,expected,required_terms
0,What are the available flights from Riyadh to Jeddah?,flight_search,"{'source': 'flight', 'origin': 'RUH', 'destination': 'JED'}","[flight, Riyadh, Jeddah]"
1,What are the available trains from KAIA Airport to Makkah?,train_search,"{'source': 'train', 'departure_station': 'KAIA (Airport)', 'arrival_station': 'Makkah'}","[train, KAIA, Makkah]"
2,What is the cheapest way from Riyadh to Makkah?,connected_route,"{'origin_city': 'Riyadh', 'destination': 'Makkah', 'preference': 'cheapest'}","[flight, train, SAR, cost, Riyadh, Makkah]"
3,What is the fastest route from Riyadh to Makkah?,connected_route,"{'origin_city': 'Riyadh', 'destination': 'Makkah', 'preference': 'fastest'}","[flight, train, duration, minutes]"
4,What is the lowest carbon option from Riyadh to Makkah?,connected_route,"{'origin_city': 'Riyadh', 'destination': 'Makkah', 'preference': 'lowest_carbon'}","[carbon, kg, flight, train]"
5,Which HHR routes are most crowded?,crowding,"{'source': 'train', 'metric': 'Occupancy_Rate_%'}","[occupancy, crowded, %]"
6,What are the peak days for flights?,flight_peak_days,"{'source': 'flight', 'metric': 'day'}","[flight, day]"


In [ ]:
# ============================================================
# 14. Retrieval relevance rules
# ============================================================
# Relevance is evaluated against the improved evidence design.
# For route questions, the relevant unit is now a complete connected-route document, not isolated flight or train rows.

def is_relevant_retrieval(result, eval_item):
    """Assign deterministic relevance labels using dataset metadata and query intent."""
    intent = eval_item["intent"]
    meta = result["metadata"]
    source = result["source"]

    if intent == "flight_search":
        exp = eval_item["expected"]
        return source == "flight" and meta.get("origin") == exp["origin"] and meta.get("destination") == exp["destination"]

    if intent == "train_search":
        exp = eval_item["expected"]
        return (
            source == "train" and
            meta.get("departure_station") == exp["departure_station"] and
            meta.get("arrival_station") == exp["arrival_station"]
        )

    if intent == "connected_route":
        exp = eval_item["expected"]
        return (
            source == "route" and
            meta.get("origin_city") == exp["origin_city"] and
            meta.get("destination") == exp["destination"] and
            meta.get("preference_label") == exp["preference"]
        )

    if intent == "crowding":
        return source == "train"

    if intent == "flight_peak_days":
        return source == "flight"

    return False


def count_total_relevant(eval_item):
    """Count all relevant documents in the indexed corpus for recall@k."""
    count = 0
    for doc in documents:
        fake_result = {"source": doc["source"], "metadata": doc["metadata"]}
        if is_relevant_retrieval(fake_result, eval_item):
            count += 1
    return count


def evaluate_retrieval(eval_items, top_k=10):
    """Evaluate retrieval with Precision@K, Recall@K, and Mean Reciprocal Rank."""
    rows = []
    for item in eval_items:
        results = semantic_retrieve(item["question"], top_k=top_k)
        relevance = [is_relevant_retrieval(result, item) for result in results]
        total_relevant = count_total_relevant(item)
        relevant_retrieved = sum(relevance)

        precision_at_k = relevant_retrieved / top_k if top_k else 0
        recall_at_k = relevant_retrieved / total_relevant if total_relevant else np.nan

        reciprocal_rank = 0.0
        for idx, is_relevant in enumerate(relevance, start=1):
            if is_relevant:
                reciprocal_rank = 1 / idx
                break

        rows.append({
            "question": item["question"],
            "intent": item["intent"],
            "top_k": top_k,
            "relevant_in_top_k": relevant_retrieved,
            "total_relevant_in_corpus": total_relevant,
            "precision_at_k": round(precision_at_k, 3),
            "recall_at_k": round(recall_at_k, 3) if not np.isnan(recall_at_k) else np.nan,
            "mrr": round(reciprocal_rank, 3),
            "top_doc_ids": [result["doc_id"] for result in results],
            "top_sources": [result["source"] for result in results],
        })

    return pd.DataFrame(rows)


retrieval_eval_df = evaluate_retrieval(evaluation_queries, top_k=20)
display(retrieval_eval_df)

print("Average Precision@10:", round(retrieval_eval_df["precision_at_k"].mean(), 3))
print("Average Recall@10:", round(retrieval_eval_df["recall_at_k"].mean(), 3))
print("Mean Reciprocal Rank:", round(retrieval_eval_df["mrr"].mean(), 3))

,question,intent,top_k,relevant_in_top_k,total_relevant_in_corpus,precision_at_k,recall_at_k,mrr,top_doc_ids,top_sources
0,What are the available flights from Riyadh to Jeddah?,flight_search,20,20,40,1.00,0.500,1.0,"[flight_1510, flight_1511, flight_1521, flight_1514, flight_1529, flight_1536, flight_1515, flight_1520, flight_1517, flight_1507, fligh...","[flight, flight, flight, flight, flight, flight, flight, flight, flight, flight, flight, flight, flight, flight, flight, flight, flight,..."
1,What are the available trains from KAIA Airport to Makkah?,train_search,20,20,146,1.00,0.137,1.0,"[train_925, train_685, train_945, train_923, train_919, train_939, train_975, train_651, train_913, train_283, train_615, train_633, tra...","[train, train, train, train, train, train, train, train, train, train, train, train, train, train, train, train, train, train, train, tr..."
2,What is the cheapest way from Riyadh to Makkah?,connected_route,20,15,15,0.75,1.000,1.0,"[route_riyadh_makkah_cheapest_1, route_riyadh_makkah_cheapest_6, route_riyadh_makkah_cheapest_7, route_riyadh_makkah_cheapest_13, route_...","[route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, ro..."
3,What is the fastest route from Riyadh to Makkah?,connected_route,20,15,15,0.75,1.000,1.0,"[route_riyadh_makkah_fastest_1, route_riyadh_makkah_fastest_11, route_riyadh_makkah_fastest_14, route_riyadh_makkah_fastest_15, route_ri...","[route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, ro..."
4,What is the lowest carbon option from Riyadh to Makkah?,connected_route,20,15,15,0.75,1.000,1.0,"[route_riyadh_makkah_lowest_carbon_4, route_riyadh_makkah_lowest_carbon_6, route_riyadh_makkah_lowest_carbon_9, route_riyadh_makkah_lowe...","[route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, route, ro..."
5,Which HHR routes are most crowded?,crowding,20,0,1350,0.00,0.000,0.0,[],[]
6,What are the peak days for flights?,flight_peak_days,20,20,1810,1.00,0.011,1.0,"[flight_574, flight_616, flight_1607, flight_1656, flight_1367, flight_1370, flight_1183, flight_330, flight_474, flight_374, flight_104...","[flight, flight, flight, flight, flight, flight, flight, flight, flight, flight, flight, flight, flight, flight, flight, flight, flight,..."


Average Precision@10: 0.75
Average Recall@10: 0.521
Mean Reciprocal Rank: 0.857


In [ ]:
# ============================================================
# 15. Structured route evaluation
# ============================================================

route_eval_cases = [
    ("Riyadh", "Makkah", "cheapest"),
    ("Riyadh", "Makkah", "fastest"),
    ("Riyadh", "Makkah", "lowest_carbon"),
    ("Dammam", "Makkah", "cheapest"),
    ("Madinah", "Makkah", "fastest"),
]

route_rows = []
for origin, destination, preference in route_eval_cases:
    route, explanation = recommend_connected_route(origin, destination, preference)
    passed = route is not None
    if route is not None:
        checks = {
            "has_flight_id": pd.notna(route.get("flight_id")),
            "has_train_trip_id": pd.notna(route.get("train_trip_id")),
            "valid_wait_time": route.get("wait_mins", -1) >= 45,
            "has_total_cost": route.get("total_cost_sar", 0) > 0,
            "has_total_duration": route.get("total_duration_mins", 0) > 0,
            "has_carbon": route.get("total_carbon_kg", 0) > 0,
        }
        passed = all(checks.values())
    else:
        checks = {}

    route_rows.append({
        "origin": origin,
        "destination": destination,
        "preference": preference,
        "passed": passed,
        "checks": checks,
        "explanation": explanation,
    })

route_eval_df = pd.DataFrame(route_rows)
display(route_eval_df)
print("Structured route pass rate:", round(route_eval_df["passed"].mean(), 3))

,origin,destination,preference,passed,checks,explanation
0,Riyadh,Makkah,cheapest,True,"{'has_flight_id': True, 'has_train_trip_id': True, 'valid_wait_time': True, 'has_total_cost': True, 'has_total_duration': True, 'has_car...","Recommended connected route from Riyadh to Makkah: take flight XY 19 with Flynas from RUH to JED, departing at 13:10 and landing at 15:0..."
1,Riyadh,Makkah,fastest,True,"{'has_flight_id': True, 'has_train_trip_id': True, 'valid_wait_time': True, 'has_total_cost': True, 'has_total_duration': True, 'has_car...","Recommended connected route from Riyadh to Makkah: take flight F3 129 with flyadeal from RUH to JED, departing at 15:20 and landing at 1..."
2,Riyadh,Makkah,lowest_carbon,True,"{'has_flight_id': True, 'has_train_trip_id': True, 'valid_wait_time': True, 'has_total_cost': True, 'has_total_duration': True, 'has_car...","Recommended connected route from Riyadh to Makkah: take flight XY 19 with Flynas from RUH to JED, departing at 13:10 and landing at 15:0..."
3,Dammam,Makkah,cheapest,True,"{'has_flight_id': True, 'has_train_trip_id': True, 'valid_wait_time': True, 'has_total_cost': True, 'has_total_duration': True, 'has_car...","Recommended connected route from Dammam to Makkah: take flight XY 403 with Flynas from DMM to JED, departing at 9:35 and landing at 11:4..."
4,Madinah,Makkah,fastest,True,"{'has_flight_id': True, 'has_train_trip_id': True, 'valid_wait_time': True, 'has_total_cost': True, 'has_total_duration': True, 'has_car...","Recommended connected route from Madinah to Makkah: take flight SV 1425 with Saudia from MED to JED, departing at 10:20 and landing at 1..."


Structured route pass rate: 1.0


In [ ]:
# ============================================================
# 16. Answer-quality evaluation
# ============================================================


def evaluate_answer_quality(question, answer, required_terms):
    answer_lower = answer.lower()
    question_lower = question.lower()
    notes = []
    score = 0
    total = 0

    def check(condition, label):
        nonlocal score, total
        total += 1
        if condition:
            score += 1
        else:
            notes.append(label)

    check(answer.strip().lower().startswith("based on the available dataset"), "Answer does not start with required grounding phrase.")
    check(len(answer.strip()) > 50, "Answer is too short.")
    check(not any(bad in answer_lower for bad in ["i assume", "probably", "not in the dataset but", "invented"]), "Answer contains unsupported speculation.")

    for term in required_terms:
        check(term.lower() in answer_lower, f"Missing required term: {term}")

    if any(w in question_lower for w in ["cheap", "cheapest", "budget"]):
        check("sar" in answer_lower, "Cost question should mention SAR.")
    if any(w in question_lower for w in ["fast", "fastest"]):
        check(any(w in answer_lower for w in ["minute", "duration", "time"]), "Fastest question should mention time/duration.")
    if any(w in question_lower for w in ["carbon", "emission", "sustainable"]):
        check(any(w in answer_lower for w in ["carbon", "kg", "emission"]), "Carbon question should mention emissions.")
    if any(w in question_lower for w in ["crowded", "occupancy"]):
        check(any(w in answer_lower for w in ["occupancy", "%", "percent"]), "Crowding question should mention occupancy.")

    return round(score / total, 3), notes

answer_rows = []
for item in evaluation_queries:
    answer, context_pack = generate_answer(item["question"], top_k=10)
    score, notes = evaluate_answer_quality(item["question"], answer, item["required_terms"])
    answer_rows.append({
        "question": item["question"],
        "intent": item["intent"],
        "answer_quality_score": score,
        "notes": notes,
        "answer": answer,
        "retrieved_doc_ids": [r["doc_id"] for r in context_pack["semantic_results"]],
    })

answer_eval_df = pd.DataFrame(answer_rows)
display(answer_eval_df[["question", "intent", "answer_quality_score", "notes"]])
print("Average answer quality score:", round(answer_eval_df["answer_quality_score"].mean(), 3))

,question,intent,answer_quality_score,notes
0,What are the available flights from Riyadh to Jeddah?,flight_search,1.0,[]
1,What are the available trains from KAIA Airport to Makkah?,train_search,1.0,[]
2,What is the cheapest way from Riyadh to Makkah?,connected_route,1.0,[]
3,What is the fastest route from Riyadh to Makkah?,connected_route,1.0,[]
4,What is the lowest carbon option from Riyadh to Makkah?,connected_route,1.0,[]
5,Which HHR routes are most crowded?,crowding,1.0,[]
6,What are the peak days for flights?,flight_peak_days,1.0,[]


Average answer quality score: 1.0


In [ ]:
# ============================================================
# 17. Final evaluation summary and export
# ============================================================

final_summary = pd.DataFrame({
    "metric": [
        "retrieval_precision_at_10_mean",
        "retrieval_recall_at_10_mean",
        "retrieval_mrr_mean",
        "structured_route_pass_rate",
        "answer_quality_mean",
    ],
    "score": [
        round(retrieval_eval_df["precision_at_k"].mean(), 3),
        round(retrieval_eval_df["recall_at_k"].mean(), 3),
        round(retrieval_eval_df["mrr"].mean(), 3),
        round(route_eval_df["passed"].mean(), 3),
        round(answer_eval_df["answer_quality_score"].mean(), 3),
    ]
})


weights = {
    "retrieval_precision_at_10_mean": 0.25,
    "retrieval_recall_at_10_mean": 0.20,
    "retrieval_mrr_mean": 0.15,
    "structured_route_pass_rate": 0.20,
    "answer_quality_mean": 0.20,
}
weighted_score = 0
for _, row in final_summary.iterrows():
    weighted_score += row["score"] * weights[row["metric"]]

final_summary.loc[len(final_summary)] = ["overall_internal_evaluation_score", round(weighted_score, 3)]

display(final_summary)

retrieval_eval_df.to_csv("retrieval_evaluation_results.csv", index=False)
route_eval_df.to_csv("structured_route_evaluation_results.csv", index=False)
answer_eval_df.to_csv("answer_quality_evaluation_results.csv", index=False)
final_summary.to_csv("final_rag_evaluation_summary.csv", index=False)

print("Saved evaluation files:")
print("- retrieval_evaluation_results.csv")
print("- structured_route_evaluation_results.csv")
print("- answer_quality_evaluation_results.csv")
print("- final_rag_evaluation_summary.csv")

,metric,score
0,retrieval_precision_at_10_mean,0.750
1,retrieval_recall_at_10_mean,0.521
2,retrieval_mrr_mean,0.857
3,structured_route_pass_rate,1.000
4,answer_quality_mean,1.000
5,overall_internal_evaluation_score,0.820


Saved evaluation files:
- retrieval_evaluation_results.csv
- structured_route_evaluation_results.csv
- answer_quality_evaluation_results.csv
- final_rag_evaluation_summary.csv


In [ ]:

#os.environ["GROQ_API_KEY"] = "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"


In [ ]:
test_queries = [
    "What is the cheapest way from Riyadh to Makkah?",
    "What is the fastest way from Riyadh to Makkah?",
    "What is the lowest carbon route from Riyadh to Makkah?"
]

for q in test_queries:
    print("="*80)
    print("QUESTION:", q)

    answer, context_pack = generate_answer(q, top_k=10)

    print("\nANSWER:\n", answer)

    print("\nRETRIEVED DOCS:")
    if isinstance(context_pack, dict):
        for k, v in context_pack.items():
            print(k)
    else:
        for item in context_pack[:3]:
            print(item["rank"], item["doc_id"], item["source"], round(item["score"], 3))

QUESTION: What is the cheapest way from Riyadh to Makkah?

ANSWER:
 Based on the available dataset, the cheapest way from Riyadh to Makkah is:

- Take flight XY 19 with Flynas from RUH to JED, departing at 13:10 and landing at 15:00.
- Then take HHR train HHR-MK-30051 from KAIA (Airport) to Makkah, departing at 15:45 and arriving at 16:17.
- Connection wait is 45 minutes.
- Total estimated duration is 187 minutes including flight, wait, and train.
- Total estimated cost is 386.50 SAR.
- Flight carbon emission is 76000.00 kg.
- Train occupancy rate is 93.0 percent.

This route is the cheapest option among the available options, with a total cost of 386.50 SAR.

RETRIEVED DOCS:
query
context
semantic_results
structured_result
QUESTION: What is the fastest way from Riyadh to Makkah?

ANSWER:
 Based on the available dataset, the fastest way from Riyadh to Makkah is:

- Take flight F3 129 with flyadeal from RUH to JED, departing at 15:20 and landing at 17:00.
- Then take HHR train HHR-MK-31

In [ ]:
answer, context_pack = generate_answer(
    "Which HHR route is the most crowded?",
    top_k=10
)

print(answer)

Based on the available dataset, the most crowded HHR (High-Speed Rail) route is between Makkah and Madinah with an average occupancy rate of 82.90%.


In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

def chatbot(query):
    answer, _ = generate_answer(query, top_k=10)
    return answer

demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(label="Ask a travel question"),
    outputs=gr.Textbox(label="Answer"),
    title="Smart Travel RAG Assistant"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://06158607e5edc42a55.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
